## Quality Checks
### **Goal:** validate raw data quality before building the staging and mart layers.

In [1]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

con = duckdb.connect()

RAW = 'data/raw'

TABLES = {
    'students':    f'{RAW}/students.csv',
    'courses':     f'{RAW}/courses.csv',
    'instructors': f'{RAW}/instructors.csv',
    'enrollments': f'{RAW}/enrollments.csv',
    'payments':    f'{RAW}/payments.csv',
}

def q(sql):
    return con.sql(sql).df()


##  NULL Checks

Check required fields for NULL values.

In [2]:
print('TABLE "payments"')

q(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN payment_id IS NULL THEN 1 ELSE 0 END) AS null_payment_id,
    SUM(CASE WHEN student_id IS NULL THEN 1 ELSE 0 END) AS null_student_id,
    
    -- course_id ALWAYS must be filled in
    SUM(CASE WHEN course_id  IS NULL OR course_id = '' THEN 1 ELSE 0 END) AS null_course_id,
    SUM(CASE WHEN amount     IS NULL THEN 1 ELSE 0 END) AS null_amount,
    SUM(CASE WHEN status     IS NULL THEN 1 ELSE 0 END) AS null_status,
    SUM(CASE WHEN date       IS NULL THEN 1 ELSE 0 END) AS null_date
FROM '{TABLES['payments']}'
""")


TABLE "payments"


,total_rows,null_payment_id,null_student_id,null_course_id,null_amount,null_status,null_date
0,10462,0.00,0.00,0.00,0.00,0.00,0.00


In [3]:
print('TABLE "students"')
q(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN student_id          IS NULL THEN 1 ELSE 0 END) AS null_student_id,
    SUM(CASE WHEN reg_date            IS NULL THEN 1 ELSE 0 END) AS null_reg_date,
    SUM(CASE WHEN country             IS NULL THEN 1 ELSE 0 END) AS null_country,
    SUM(CASE WHEN acquisition_channel IS NULL THEN 1 ELSE 0 END) AS null_channel
FROM '{TABLES['students']}'
""")

TABLE "students"


,total_rows,null_student_id,null_reg_date,null_country,null_channel
0,4881,0.00,0.00,0.00,0.00


In [4]:
print('TABLE "enrollments"')
q(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN enrollment_id  IS NULL THEN 1 ELSE 0 END) AS null_enrollment_id,
    SUM(CASE WHEN student_id     IS NULL THEN 1 ELSE 0 END) AS null_student_id,
    SUM(CASE WHEN course_id      IS NULL THEN 1 ELSE 0 END) AS null_course_id,
    SUM(CASE WHEN enroll_date    IS NULL THEN 1 ELSE 0 END) AS null_enroll_date,
    SUM(CASE WHEN completion_pct IS NULL THEN 1 ELSE 0 END) AS null_completion_pct
FROM '{TABLES['enrollments']}'
""")

TABLE "enrollments"


,total_rows,null_enrollment_id,null_student_id,null_course_id,null_enroll_date,null_completion_pct
0,10462,0.00,0.00,0.00,0.00,0.00


## Duplicate Checks  

**1. Checking primary keys for duplicates:**

In [5]:
pk_checks = [
    ('students',    'student_id'),
    ('courses',     'course_id'),
    ('instructors', 'instructor_id'),
    ('enrollments', 'enrollment_id'),
    ('payments',    'payment_id'),
]

for table, pk in pk_checks:
    path = TABLES[table]
    result = con.sql(f"""
        SELECT COUNT(*) - COUNT(DISTINCT {pk}) AS duplicate_pks
        FROM '{path}'
    """).df()
    val = result.iloc[0, 0]
    print(f'   {table}.{pk} — duplicates: {val}')

   students.student_id — duplicates: 0
   courses.course_id — duplicates: 0
   instructors.instructor_id — duplicates: 0
   enrollments.enrollment_id — duplicates: 0
   payments.payment_id — duplicates: 0


**2. One student should not be enrolled in the same course twice:**

In [6]:
result = q(f"""
SELECT COUNT(*) AS duplicate_pairs
FROM (
    SELECT student_id, course_id, COUNT(*) AS cnt
    FROM '{TABLES['enrollments']}'
    GROUP BY student_id, course_id
    HAVING COUNT(*) > 1
) t
""")
val = result.iloc[0, 0]


print(f'  {"There are no duplicates" if val == 0 else f" {val} duplicate pairs"}')

  There are no duplicates


**3. One student - one payment for one course:**

In [7]:
result = q(f"""
SELECT COUNT(*) AS duplicate_pairs
FROM (
    SELECT student_id, course_id, COUNT(*) AS cnt
    FROM '{TABLES['payments']}'
    GROUP BY student_id, course_id
    HAVING COUNT(*) > 1
) t
""")
val = result.iloc[0, 0]
print(f'  {"There are no duplicates" if val == 0 else f" {val} duplicate pairs"}')

  There are no duplicates
